# 06. Results

This notebook only reads what `src/make_report.py` wrote to `results/` and the run directories in
`runs/`. Run `python src/make_report.py --runs runs --results results` first.

Protocol (section 6.5 of `docs/AGENT_GUIDE.md`): every comparison and ablation is shown on **val**.
The final LoRA run was chosen by val exact match and written to `results/selection.json` before any
test number was read. Test numbers appear only for the headline runs (zero-shot, 3-shot, the
selected LoRA run, and the large zero-shot model if it ran). Test numbers of all other runs are in an
appendix marked as not used for selection.

Every figure cell checks that its runs exist and prints a message instead of failing when they do not.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

RUNS = Path("../runs")
RESULTS = Path("../results")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "axes.grid.axis": "y", "grid.alpha": 0.3, "axes.axisbelow": True})
COLORS = {"zero-shot": "#2a78d6", "3-shot": "#eb6834", "selected": "#1baf7a", "33b zero-shot": "#eda100",
          "lora": "#1baf7a", "qlora": "#6250d6"}
MAIN = "#2a78d6"


def read_csv(name):
    path = RESULTS / name
    return pd.read_csv(path) if path.exists() else None


def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


def bars(ax, labels, means, lows=None, highs=None, colors=MAIN):
    """Bars with 95% intervals where they exist."""
    lows = lows if lows is not None else [None] * len(means)
    highs = highs if highs is not None else [None] * len(means)
    errors = [[float("nan") if pd.isna(l) else m - l for m, l in zip(means, lows)],
              [float("nan") if pd.isna(h) else h - m for m, h in zip(means, highs)]]
    ax.bar(range(len(means)), means, color=colors, width=0.6, yerr=errors, capsize=4,
           error_kw={"elinewidth": 1, "ecolor": "#52514e"})
    ax.set_xticks(range(len(means)), labels)


def save(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURES / name, dpi=150, bbox_inches="tight")
    plt.show()


selection = read_json(RESULTS / "selection.json")
headline = read_json(RESULTS / "headline.json")
results = read_csv("results.csv")
by_pattern = read_csv("by_pattern.csv")
detection = read_csv("detection.csv")
costs = read_csv("costs.csv")
SELECTED = selection["selected_run"] if selection else None
val = results[results["split"] == "val"].set_index("run") if results is not None else None
print("selected run:", SELECTED)

## Val table and selection

All repair runs on val. `used_for_selection` marks the LoRA runs that competed for selection (seed
replicates and the other data tier do not compete). The selected run is the one with the highest val
exact match among them.

In [ ]:
if results is None:
    print("Skipped: results/results.csv is missing; run src/make_report.py first.")
else:
    shown = ["tier", "model", "method", "rank", "quantized", "n", "exact_match", "exact_match_low",
             "exact_match_high", "localized", "direction_correct", "parse_ok", "changed", "bleu", "copy_bleu",
             "used_for_selection", "selected"]
    display(val[shown].sort_values(["tier", "exact_match"], ascending=[True, False]).round(4))
    print(json.dumps({k: selection[k] for k in ("selected_run", "criterion", "tier", "candidates", "note")},
                     indent=2))

## Figure 1. Headline: test

Exact match, localization and fix-direction accuracy on the test pairs for zero-shot, 3-shot, the
selected LoRA run and, if it ran, the large zero-shot model. Error bars are 95% bootstrap intervals
over pairs. The table under the figure gives the paired bootstrap difference "selected minus other"
on the pairs both runs predicted; an interval that does not include zero is a difference the test
set can resolve.

In [ ]:
if not headline or not headline["rows"]:
    print("Skipped: results/headline.json has no rows (no selected LoRA run yet).")
else:
    rows = headline["rows"]
    metrics = [m for m in ("exact_match", "localized", "direction_correct") if any(m in r for r in rows)]
    fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.8), squeeze=False)
    for ax, metric in zip(axes[0], metrics):
        shown = [r for r in rows if metric in r]
        bars(ax, [f"{r['role']}\n{r['run']}" for r in shown], [r[metric]["mean"] for r in shown],
             [r[metric].get("low") for r in shown], [r[metric].get("high") for r in shown],
             [COLORS[r["role"]] for r in shown])
        ax.set_title(f"{metric} (test)")
        ax.tick_params(axis="x", labelsize=8)
        if metric == "direction_correct":
            ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1)
    save(fig, "headline_test.png")
    differences = []
    for c in headline["comparisons"]:
        for metric in ("exact_match", "localized", "direction_correct"):
            if metric in c:
                differences.append({"comparison": c["label"], "other run": c["b"], "metric": metric,
                                    "pairs": c[metric]["n"], "difference": c[metric]["mean"],
                                    "low": c[metric]["low"], "high": c[metric]["high"]})
        if "missing" in c:
            print(c["label"], "-", c["missing"])
    display(pd.DataFrame(differences).round(4))

## Figure 2. LoRA rank (val)

Val exact match of the main model with LoRA rank 8, 16 and 32, all other settings equal. Error bars are
95% bootstrap intervals.

In [ ]:
RANK_RUNS = ["lora_6.7b_r8", "lora_6.7b_r16", "lora_6.7b_r32"]
present = [r for r in RANK_RUNS if val is not None and r in val.index and pd.notna(val.loc[r, "exact_match"])]
if len(present) < 2:
    print("Skipped: fewer than two of", RANK_RUNS, "have val results.")
else:
    rows = val.loc[present]
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    bars(ax, [f"r = {int(r)}" if pd.notna(r) else name for name, r in zip(present, rows["rank"])],
         rows["exact_match"].tolist(), rows["exact_match_low"].tolist(), rows["exact_match_high"].tolist())
    ax.set_ylabel("exact match (val)")
    ax.set_title("LoRA rank")
    save(fig, "rank_ablation_val.png")

## Figure 3. Model size and quantization (val)

Val exact match for the small and the main model, each zero-shot and with LoRA, plus QLoRA (4-bit base
model) on the main model and the large zero-shot model if it ran. Colour shows the method.

In [ ]:
SIZE_RUNS = ["zs_1.3b", "lora_1.3b_r16", "zs_6.7b", "lora_6.7b_r16", "qlora_6.7b_r16", "zs_33b"]
present = [r for r in SIZE_RUNS if val is not None and r in val.index and pd.notna(val.loc[r, "exact_match"])]
if len(present) < 2:
    print("Skipped: fewer than two of", SIZE_RUNS, "have val results.")
else:
    rows = val.loc[present]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    bars(ax, present, rows["exact_match"].tolist(), rows["exact_match_low"].tolist(),
         rows["exact_match_high"].tolist(), [COLORS.get(m, MAIN) for m in rows["method"]])
    ax.legend(handles=[Patch(color=COLORS.get(m, MAIN), label=m) for m in dict.fromkeys(rows["method"])],
              frameon=False)
    ax.set_ylabel("exact match (val)")
    ax.set_title("Model size and quantization")
    ax.tick_params(axis="x", labelsize=8)
    save(fig, "size_quantization_val.png")

## Figure 4. Exact match per bug type (test)

Exact match per `sstub_pattern` for the zero-shot run and the selected LoRA run of the same model.
Bug types with too few test pairs are grouped as OTHER (same rule as `repair_utils.summarize`). The
number of pairs per type is in the table below the figure.

In [ ]:
roles = ["zero-shot", "selected"]
if by_pattern is None or not set(roles) <= set(by_pattern["role"]):
    print("Skipped: by_pattern.csv needs both a zero-shot and a selected run.")
else:
    table = by_pattern[by_pattern["role"].isin(roles)].pivot(index="sstub_pattern", columns="role",
                                                             values="exact_match")[roles]
    table = table.sort_values("selected")
    fig, ax = plt.subplots(figsize=(7, 0.35 * len(table) + 1.5))
    height = 0.38
    for i, role in enumerate(roles):
        ax.barh([y + (i - 0.5) * height for y in range(len(table))], table[role], height=height * 0.9,
                color=COLORS[role], label=role)
    ax.set_yticks(range(len(table)), table.index, fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("exact match (test)")
    ax.legend(frameon=False, loc="lower right", reverse=True)
    save(fig, "by_pattern_test.png")
    display(by_pattern[by_pattern["role"].isin(roles)].pivot(index="sstub_pattern", columns="role",
                                                             values=["n", "exact_match", "localized"]).round(4))

## Figure 5. Training and validation loss of the selected run

Training loss (logged every few optimizer steps) and validation loss on the fixed val subset used
during training. The adapter kept is the checkpoint with the lowest validation loss.

In [ ]:
metrics = read_json(RUNS / str(SELECTED) / "metrics.json") if SELECTED else None
history = pd.DataFrame(metrics.get("log_history") or []) if metrics else pd.DataFrame()
if history.empty or "step" not in history:
    print("Skipped: no log_history for the selected run", SELECTED)
else:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for column, color, label in (("loss", MAIN, "train loss"), ("eval_loss", COLORS["3-shot"], "val loss")):
        if column in history:
            part = history.dropna(subset=[column])
            ax.plot(part["step"], part[column], color=color, linewidth=2, marker="o" if column == "eval_loss" else None,
                    markersize=5, label=label)
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("loss")
    ax.set_title(f"{SELECTED}")
    ax.legend(frameon=False)
    print("best checkpoint:", metrics.get("best_checkpoint"), "| stopped by time budget:",
          metrics.get("stopped_by_time_budget"))
    save(fig, "loss_curve.png")

## Figure 6. Detection (val)

Accuracy of the isolated-snippet detectors (TF-IDF, BiLSTM, the LLM zero-shot and with LoRA), which
see one snippet and say "buggy" or "fixed", next to fix-direction accuracy of the headline repair
runs, which see both versions of a pair and say which one is the fix. The dashed line is chance.

In [ ]:
if detection is None or detection.empty:
    print("Skipped: detection.csv is missing or empty.")
else:
    headline_runs = {r["run"] for r in headline["rows"]} if headline else set()
    shown = detection[(detection["kind"] == "isolated snippet") | detection["run"].isin(headline_runs)]
    shown = shown.dropna(subset=["accuracy"])
    if shown.empty:
        print("Skipped: no detection accuracy to show.")
    else:
        labels = [s if k == "isolated snippet" else f"fix direction\n{r}"
                  for s, k, r in zip(shown["system"], shown["kind"], shown["run"])]
        colors = [MAIN if k == "isolated snippet" else COLORS["selected"] for k in shown["kind"]]
        fig, ax = plt.subplots(figsize=(1.2 * len(shown) + 2, 3.8))
        bars(ax, labels, shown["accuracy"].tolist(), shown["accuracy_low"].tolist(),
             shown["accuracy_high"].tolist(), colors)
        ax.axhline(0.5, color="#52514e", linestyle="--", linewidth=1)
        ax.legend(handles=[Patch(color=MAIN, label="one snippet: buggy or fixed"),
                           Patch(color=COLORS["selected"], label="pair: which version is the fix")],
                  frameon=False, loc="upper left", fontsize=8)
        ax.set_ylabel("accuracy (val)")
        ax.tick_params(axis="x", labelsize=8)
        save(fig, "detection_val.png")
        display(shown.round(4))

## Appendix: test numbers of runs not used for selection

For completeness only. None of these numbers was used to choose anything.

In [ ]:
if results is None:
    print("Skipped: results/results.csv is missing.")
else:
    appendix = results[results["section"] == "test appendix (not used for selection)"]
    display(appendix[["run", "tier", "method", "rank", "n", "exact_match", "exact_match_low", "exact_match_high",
                      "localized", "direction_correct"]].round(4))

## Error analysis (selected run, test)

A fixed-seed sample of test pairs of the selected run, the same number from each group:

- **fixed correctly**: the output equals the developer fix;
- **right line, wrong fix**: the model edited a line the developer edited, but the result differs;
- **wrong line or copied input**: the model edited other lines, or returned the input unchanged.

Each example shows the buggy snippet, the developer fix and the model output. The one-line notes are
written by the students: put them in the `notes` dictionary below (pair id to note).

In [ ]:
SEED = 42
PER_GROUP = 5
PLACEHOLDER = "TODO (students): one-line note"
notes = {}  # pair_id -> one-line note, written by the students

path = RUNS / str(SELECTED) / "test_predictions.parquet"
if not SELECTED or not path.exists():
    print("Skipped: no test predictions for the selected run", SELECTED)
    sample = pd.DataFrame()
else:
    predictions = pd.read_parquet(path)
    exact = predictions["exact_match"].astype(bool)
    localized = predictions["localized"].astype(bool)
    groups = {"fixed correctly": exact, "right line, wrong fix": localized & ~exact,
              "wrong line or copied input": ~localized & ~exact}
    parts = []
    for name, mask in groups.items():
        rows = predictions[mask]
        parts.append(rows.sample(min(PER_GROUP, len(rows)), random_state=SEED).assign(group=name))
        if len(rows) < PER_GROUP:
            print(f"group '{name}' has only {len(rows)} pairs")
    sample = pd.concat(parts)[["group", "pair_id", "sstub_pattern", "changed", "buggy", "fixed", "prediction"]]
    sample["note"] = [notes.get(int(pid), PLACEHOLDER) for pid in sample["pair_id"]]
    sample.to_csv(RESULTS / "error_analysis.csv", index=False)
    display(sample[["group", "pair_id", "sstub_pattern", "changed", "note"]])

In [ ]:
for row in sample.itertuples():
    print(f"=== {row.group} | pair {row.pair_id} | {row.sstub_pattern} | note: {row.note}")
    for title, text in (("buggy", row.buggy), ("developer fix", row.fixed), ("model output", row.prediction)):
        print(f"--- {title}")
        print(text)
    print()

## Cost table

Per run: trainable parameters, their share of all parameters, adapter size on disk, training time and
training throughput. Untrained runs (zero-shot, 3-shot) have no training cost and show empty cells.

In [ ]:
if costs is None or costs.empty:
    print("Skipped: costs.csv is missing or empty.")
else:
    display(costs[costs["trained"]].drop(columns=["trained"]).round(4))